# 05 — Public Soils Availability and Coverage

This notebook audits public soil-data availability. It does **not** assume that Oglala Lakota soils are available, and it does not use adjacent-county soils as a substitute. A missing catalog record documents public availability status only; it does not establish why data are absent.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"src").is_dir())
sys.path.insert(0, str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
import pandas as pd
import geopandas as gpd
import yaml
from IPython.display import display
from src.constants import REPO_ROOT as ROOT, OUTPUTS_DIR
from src.loaders import load_tribal_boundaries
from src.sovereignty import print_data_acknowledgment, generate_citations
with open(ROOT/"config"/"config.yaml") as stream: CONFIG = yaml.safe_load(stream)
primary = load_tribal_boundaries(["Pine Ridge"])
pine_ridge = primary[primary["NAME"] == "Pine Ridge"]

In [ ]:
print_data_acknowledgment(["usda_ssurgo", "census_aiannh"])

## Local inventory

`.ppkx` watershed projects are reported but rejected as SSURGO sources. Only actual soil geodatabases enter coverage assessment.

In [ ]:
from src.loaders import ssurgo_inventory, load_ssurgo_mapunits
inventory = ssurgo_inventory()
display(inventory)
mapunits = load_ssurgo_mapunits()
print(f"Usable local SSURGO polygons: {len(mapunits):,}")

## Reproducible public-catalog check

Set `RUN_LIVE_AUDIT=True` only when intentionally contacting USDA. Record the timestamp, endpoint, requested identifiers, and exact machine-observed status. `not_listed` and request failure are different outcomes; neither establishes causation.

In [ ]:
from datetime import datetime, timezone
from src.ssurgo import SDA_TABULAR_URL, SDAError, survey_status
RUN_LIVE_AUDIT = False
candidates = sorted({s for group in CONFIG["ssurgo_areasymbol"].values() for s in group})
audit = pd.DataFrame({"requested_areasymbol": candidates})
audit["checked_at_utc"] = pd.NA
audit["endpoint"] = SDA_TABULAR_URL
audit["machine_status"] = "not_checked_this_run"
if RUN_LIVE_AUDIT:
    checked = datetime.now(timezone.utc).isoformat()
    try:
        live = survey_status(candidates).rename(columns={"areasymbol": "requested_areasymbol"})
        audit = audit.drop(columns=["machine_status"]).merge(live, on="requested_areasymbol", how="left")
        audit["checked_at_utc"] = checked
        audit["endpoint"] = SDA_TABULAR_URL
    except SDAError as exc:
        audit["checked_at_utc"] = checked
        audit["machine_status"] = f"request_failed: {exc}"
display(audit)
if RUN_LIVE_AUDIT:
    print("Interpret only the recorded machine status; absence and cause are separate questions.")
else:
    print("No current availability conclusion: the live audit was not run.")

## Geographic coverage gate

Analytical coverage is measured against each reservation polygon. The analysis stops below the configured threshold; nearby polygons may be displayed only as explicitly labeled regional context.

In [ ]:
from src.soil_evidence import assess_coverage, require_coverage, SoilCoverageError
reports = [assess_coverage(mapunits, pine_ridge, "Pine Ridge public SSURGO")]
display(pd.DataFrame([r.__dict__ for r in reports]))
try:
    require_coverage(mapunits, pine_ridge, "Pine Ridge public SSURGO")
    print("Pine Ridge public-soils analysis authorized by coverage gate.")
except SoilCoverageError as exc:
    print(exc)
    print("Result: Pine Ridge soil attributes remain UNKNOWN; adjacent surveys are context only.")

## Defensible conclusion

> Oglala Lakota soils data were not obtained through the tested public USDA pathways as of the recorded checks.

Cause is unconfirmed. Legitimate next routes are Nation-authorized access, a Nation-to-Nation/institutional agreement with NRCS, or Tribal-owned field collection. This notebook does not interpolate surrounding counties into the missing geography.